<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/vertex_ai_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vertex AI Hyperparameter Sweep

Submit hyperparameter tuning jobs to Vertex AI using `sweep.py`.

**Modes:**
- **Single-stage** (`launch`) — Sweep one curriculum stage at a time
- **All stages** (`launch-all`) — Sweep stages 1→2→3 end-to-end, automatically chaining the best checkpoint from each stage to the next

**What this notebook does:**
1. Authenticates with Google Cloud
2. Builds and pushes the training Docker image to Artifact Registry
3. Submits a Vertex AI Hyperparameter Tuning job via the Python SDK

**Prerequisites:**
- A Google Cloud project with billing enabled
- Vertex AI and Artifact Registry APIs enabled
- A GCS bucket for training artifacts

## 1. Setup & Authentication

In [ ]:
# Install the Vertex AI SDK (required for job submission)
!pip install -q google-cloud-aiplatform

In [ ]:
# Authenticate with Google Cloud
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Google Cloud.")
else:
    print("Not running in Colab — ensure `gcloud auth login` and `gcloud auth application-default login` are done.")

## 2. Configuration

Fill in your GCP project settings below. The `IMAGE_URI` will be set automatically after building the Docker image (Section 3), or you can set it manually if you already have an image pushed.

In [ ]:
# ── GCP project settings ─────────────────────────────────────────────────────
PROJECT_ID = "your-gcp-project-id"   # @param {type:"string"}
REGION = "us-central1"               # @param {type:"string"}
BUCKET = "your-gcs-bucket"           # @param {type:"string"} — without gs:// prefix

# ── Docker image ─────────────────────────────────────────────────────────────
# Set this after building (Section 3), or manually if you already have an image.
REPO_NAME = "mesozoic-labs"
IMAGE_TAG = "latest"
IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/trainer:{IMAGE_TAG}"

# ── Sweep settings ────────────────────────────────────────────────────────────
SPECIES = "velociraptor"  # @param ["velociraptor", "brachiosaurus", "trex"]
ALGORITHM = "ppo"         # @param ["ppo", "sac"]

# ── Machine settings ──────────────────────────────────────────────────────────
MACHINE_TYPE = "n1-standard-8"       # @param {type:"string"}
ACCELERATOR_TYPE = "NVIDIA_TESLA_T4" # @param ["NVIDIA_TESLA_T4", "NVIDIA_TESLA_V100", "NVIDIA_A100_80GB"]
ACCELERATOR_COUNT = 1                # @param {type:"integer"}

# ── HPT budget ────────────────────────────────────────────────────────────────
MAX_TRIALS = 20    # @param {type:"integer"} — total trials per stage
PARALLEL_TRIALS = 5 # @param {type:"integer"} — concurrent trials
N_ENVS = 4          # @param {type:"integer"} — parallel envs per trial worker

# ── Optional: W&B logging ─────────────────────────────────────────────────────
WANDB_API_KEY = ""  # @param {type:"string"} — leave empty to disable

print(f"Project:  {PROJECT_ID}")
print(f"Region:   {REGION}")
print(f"Bucket:   gs://{BUCKET}")
print(f"Image:    {IMAGE_URI}")
print(f"Species:  {SPECIES}")
print(f"Algorithm: {ALGORITHM}")
print(f"Trials:   {MAX_TRIALS} (parallel: {PARALLEL_TRIALS})")

## 3. Build & Push Docker Image

This section clones the repo, builds the training container, and pushes it to Artifact Registry. **Skip this section if you already have an image pushed** (e.g. from running `scripts/setup_vertex_ai.sh`).

In [ ]:
# Enable required APIs (idempotent)
!gcloud services enable aiplatform.googleapis.com artifactregistry.googleapis.com \
    --project={PROJECT_ID} --quiet
print("APIs enabled.")

In [ ]:
# Create Artifact Registry repository (idempotent)
!gcloud artifacts repositories create {REPO_NAME} \
    --repository-format=docker \
    --location={REGION} \
    --description="Mesozoic Labs training containers" \
    --project={PROJECT_ID} 2>/dev/null || echo "Repository already exists."

# Configure Docker authentication
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

In [ ]:
import pathlib
import subprocess

REPO_DIR = pathlib.Path("/content/mesozoic-labs")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(REPO_DIR)],
        check=True,
    )
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

# Build the Docker image
print(f"\nBuilding image: {IMAGE_URI}")
!cd /content/mesozoic-labs && docker build -t {IMAGE_URI} .

In [ ]:
# Push the image to Artifact Registry
!docker push {IMAGE_URI}
print(f"\nImage pushed: {IMAGE_URI}")

## 4. Initialise Vertex AI SDK

In [ ]:
from google.cloud import aiplatform
from google.cloud.aiplatform import hyperparameter_tuning as hpt

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=f"gs://{BUCKET}",
)
print(f"Vertex AI initialised (project={PROJECT_ID}, region={REGION})")

## 5. Search Space

Define the hyperparameters to tune. The defaults below match `sweep.py`. Edit as needed.

**Naming convention:** `{algo}_{param}` (e.g. `ppo_learning_rate`, `sac_batch_size`). These are automatically converted to `--override ppo.learning_rate=X` inside each trial worker.

In [ ]:
# ── PPO search space (used when ALGORITHM == "ppo") ────────────────────────
PPO_PARAMETER_SPEC = {
    "ppo_learning_rate": hpt.DoubleParameterSpec(min=1e-5, max=3e-4, scale="log"),
    "ppo_ent_coef": hpt.DoubleParameterSpec(min=1e-4, max=0.05, scale="log"),
    "ppo_batch_size": hpt.DiscreteParameterSpec(values=[64, 128, 256, 512], scale="linear"),
    "ppo_gamma": hpt.DoubleParameterSpec(min=0.97, max=0.999, scale="linear"),
    "ppo_n_steps": hpt.DiscreteParameterSpec(values=[1024, 2048, 4096], scale="linear"),
}

# ── SAC search space (used when ALGORITHM == "sac") ────────────────────────
SAC_PARAMETER_SPEC = {
    "sac_learning_rate": hpt.DoubleParameterSpec(min=1e-5, max=3e-4, scale="log"),
    "sac_batch_size": hpt.DiscreteParameterSpec(values=[128, 256, 512], scale="linear"),
    "sac_gamma": hpt.DoubleParameterSpec(min=0.97, max=0.999, scale="linear"),
}

PARAMETER_SPEC = PPO_PARAMETER_SPEC if ALGORITHM == "ppo" else SAC_PARAMETER_SPEC

print(f"Search space for {ALGORITHM.upper()}:")
for name in PARAMETER_SPEC:
    print(f"  {name}")

---

## Option A: Single-Stage Sweep (`launch`)

Submit a sweep for **one curriculum stage**. The job returns immediately (non-blocking). Use this when you want to tune one stage at a time or experiment with a custom search space per stage.

In [ ]:
# ── Single-stage settings ─────────────────────────────────────────────────────
STAGE = 1             # @param {type:"integer"} — curriculum stage (1, 2, or 3)
TIMESTEPS = 500_000   # @param {type:"integer"} — training timesteps per trial

# Optional: warm-start from a previous checkpoint (GCS-mounted path)
LOAD_PATH = ""  # @param {type:"string"} — e.g. /gcs/BUCKET/sweeps/velociraptor/stage1/1/models/stage1_final.zip

In [ ]:
# Build the trial worker args
output_base = f"/gcs/{BUCKET}/sweeps/{SPECIES}/stage{STAGE}"

trial_args = [
    "environments/shared/scripts/sweep.py",
    "trial",
    "--species", SPECIES,
    "--stage", str(STAGE),
    "--algorithm", ALGORITHM,
    "--timesteps", str(TIMESTEPS),
    "--n-envs", str(N_ENVS),
    "--output-dir", output_base,
]
if LOAD_PATH:
    trial_args += ["--load", LOAD_PATH]

env_vars = []
if WANDB_API_KEY:
    trial_args.append("--wandb")
    env_vars.append({"name": "WANDB_API_KEY", "value": WANDB_API_KEY})
    env_vars.append({"name": "WANDB_PROJECT", "value": "mesozoic-labs"})

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": MACHINE_TYPE,
            "accelerator_type": ACCELERATOR_TYPE,
            "accelerator_count": ACCELERATOR_COUNT,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "command": ["python"],
            "args": trial_args,
            **(dict(env=env_vars) if env_vars else {}),
        },
    }
]

display_name = f"{SPECIES}-stage{STAGE}-{ALGORITHM}-sweep"

custom_job = aiplatform.CustomJob(
    display_name=f"{display_name}-trial",
    worker_pool_specs=worker_pool_specs,
)

hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=display_name,
    custom_job=custom_job,
    metric_spec={"best_mean_reward": "maximize"},
    parameter_spec=PARAMETER_SPEC,
    max_trial_count=MAX_TRIALS,
    parallel_trial_count=PARALLEL_TRIALS,
)

print(f"Submitting: {display_name}")
print(f"  Trials: {MAX_TRIALS}  |  Parallel: {PARALLEL_TRIALS}")
print(f"  Timesteps/trial: {TIMESTEPS:,}")
print(f"  Output: gs://{BUCKET}/sweeps/{SPECIES}/stage{STAGE}/")
if LOAD_PATH:
    print(f"  Warm-start: {LOAD_PATH}")

hpt_job.run(sync=False)

print(f"\nJob submitted: {hpt_job.resource_name}")
print(f"Monitor: https://console.cloud.google.com/vertex-ai/training/hyperparameter-tuning-jobs?project={PROJECT_ID}")

---

## Option B: All-Stages Sweep (`launch-all`)

Sweep all three curriculum stages sequentially in a single run. Each stage waits for the previous one to finish, then automatically picks the best trial's checkpoint as the warm-start model for the next stage.

**This cell blocks until all three stages are complete**, which can take several hours depending on your timestep budgets and trial counts.

In [ ]:
# ── Per-stage timestep budgets ────────────────────────────────────────────────
TIMESTEPS_STAGE1 = 500_000    # @param {type:"integer"}
TIMESTEPS_STAGE2 = 1_000_000  # @param {type:"integer"}
TIMESTEPS_STAGE3 = 1_500_000  # @param {type:"integer"}

In [ ]:
import time

timesteps_per_stage = [TIMESTEPS_STAGE1, TIMESTEPS_STAGE2, TIMESTEPS_STAGE3]
load_path = None
all_jobs = []

for stage in range(1, 4):
    timesteps = timesteps_per_stage[stage - 1]
    output_base = f"/gcs/{BUCKET}/sweeps/{SPECIES}/stage{stage}"

    print("=" * 60)
    print(f"Stage {stage} / 3  —  {timesteps:,} timesteps/trial")
    print("=" * 60)

    trial_args = [
        "environments/shared/scripts/sweep.py",
        "trial",
        "--species", SPECIES,
        "--stage", str(stage),
        "--algorithm", ALGORITHM,
        "--timesteps", str(timesteps),
        "--n-envs", str(N_ENVS),
        "--output-dir", output_base,
    ]
    if load_path:
        trial_args += ["--load", load_path]
        print(f"  Warm-start: {load_path}")

    env_vars = []
    if WANDB_API_KEY:
        trial_args.append("--wandb")
        env_vars.append({"name": "WANDB_API_KEY", "value": WANDB_API_KEY})
        env_vars.append({"name": "WANDB_PROJECT", "value": "mesozoic-labs"})

    worker_pool_specs = [
        {
            "machine_spec": {
                "machine_type": MACHINE_TYPE,
                "accelerator_type": ACCELERATOR_TYPE,
                "accelerator_count": ACCELERATOR_COUNT,
            },
            "replica_count": 1,
            "container_spec": {
                "image_uri": IMAGE_URI,
                "command": ["python"],
                "args": trial_args,
                **(dict(env=env_vars) if env_vars else {}),
            },
        }
    ]

    display_name = f"{SPECIES}-stage{stage}-{ALGORITHM}-sweep"

    custom_job = aiplatform.CustomJob(
        display_name=f"{display_name}-trial",
        worker_pool_specs=worker_pool_specs,
    )

    hpt_job = aiplatform.HyperparameterTuningJob(
        display_name=display_name,
        custom_job=custom_job,
        metric_spec={"best_mean_reward": "maximize"},
        parameter_spec=PARAMETER_SPEC,
        max_trial_count=MAX_TRIALS,
        parallel_trial_count=PARALLEL_TRIALS,
    )

    start = time.time()
    hpt_job.run(sync=True)  # Block until this stage completes
    elapsed = time.time() - start

    print(f"\nStage {stage} complete in {elapsed / 60:.1f} min")
    print(f"  Job: {hpt_job.resource_name}")
    all_jobs.append(hpt_job)

    # Find the best trial and use its checkpoint for the next stage
    if stage < 3:
        best_trial = None
        best_value = float("-inf")
        for trial in hpt_job.trials:
            if trial.final_measurement and trial.final_measurement.metrics:
                for metric in trial.final_measurement.metrics:
                    if metric.metric_id == "best_mean_reward" and metric.value > best_value:
                        best_value = metric.value
                        best_trial = trial
        if best_trial is not None:
            load_path = f"/gcs/{BUCKET}/sweeps/{SPECIES}/stage{stage}/{best_trial.id}/models/stage{stage}_final.zip"
            print(f"  Best trial: {best_trial.id}  (reward={best_value:.2f})")
            print(f"  Next stage will load: {load_path}")
        else:
            print("  WARNING: No completed trials found — next stage starts from scratch.")
            load_path = None

print("\n" + "=" * 60)
print(f"ALL STAGES COMPLETE for {SPECIES} ({ALGORITHM.upper()})")
print(f"Results: gs://{BUCKET}/sweeps/{SPECIES}/")
print("=" * 60)

---

## 6. Monitor & Inspect Results

In [ ]:
# Check the status of the most recently submitted job
# (works for both Option A and Option B)
try:
    job_to_check = hpt_job  # from Option A
except NameError:
    job_to_check = all_jobs[-1] if all_jobs else None  # from Option B

if job_to_check:
    print(f"Job:    {job_to_check.display_name}")
    print(f"State:  {job_to_check.state}")
    print(f"Resource: {job_to_check.resource_name}")
else:
    print("No job found — run Option A or Option B first.")

In [ ]:
# Print trial results for a completed HPT job
import pandas as pd

def trials_to_dataframe(job):
    """Extract trial hyperparameters and metrics into a DataFrame."""
    rows = []
    for trial in job.trials:
        row = {"trial_id": trial.id}
        if hasattr(trial, "parameters") and trial.parameters:
            for param in trial.parameters:
                row[param.parameter_id] = param.value
        if trial.final_measurement and trial.final_measurement.metrics:
            for metric in trial.final_measurement.metrics:
                row[metric.metric_id] = metric.value
        rows.append(row)
    return pd.DataFrame(rows).sort_values("best_mean_reward", ascending=False)

if job_to_check and job_to_check.trials:
    df = trials_to_dataframe(job_to_check)
    display(df)
else:
    print("No trial results available yet.")

In [ ]:
# List sweep artifacts in GCS
!gcloud storage ls gs://{BUCKET}/sweeps/{SPECIES}/ --recursive 2>/dev/null | head -30 || echo "No artifacts found (job may still be running)."

In [ ]:
# Download the best model from a completed sweep
# Adjust STAGE and TRIAL_ID based on the results above.
DOWNLOAD_STAGE = 3   # @param {type:"integer"}
TRIAL_ID = "1"       # @param {type:"string"} — best trial ID from the results table

src = f"gs://{BUCKET}/sweeps/{SPECIES}/stage{DOWNLOAD_STAGE}/{TRIAL_ID}/models/stage{DOWNLOAD_STAGE}_final.zip"
dst = f"/content/{SPECIES}_stage{DOWNLOAD_STAGE}_best.zip"

!gcloud storage cp {src} {dst} && echo "Downloaded to {dst}" || echo "File not found: {src}"